# Task 4 — Graph Topology Ingestion into Neo4j

## Yêu cầu của đề

Kafka Connector Sink phải ghi node/edge trực tiếp từ Kafka vào Neo4j, không có
Spark trung gian, và replay không tạo trùng.

## Cách triển khai và lý do

Hai connector độc lập đọc `cpg.nodes` và `cpg.edges`. Cypher xử lý
`NODE_UPSERT`/`NODE_DELETE` và `EDGE_UPSERT`/`EDGE_DELETE`; UPSERT dùng `MERGE`
theo ID, còn DELETE dọn topology revision cũ. Constraint unique cho `node_id`
được tạo trước khi đăng ký connector.

Source:
[`neo4j-sink-nodes.json`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task4/connectors/neo4j-sink-nodes.json),
[`neo4j-sink-edges.json`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task4/connectors/neo4j-sink-edges.json),
[`register_connectors.sh`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task4/scripts/register_connectors.sh).

## Output thật đã chạy


In [1]:
import json
from pathlib import Path

proof = json.loads(Path("../artifacts/task4/neo4j_verification.json").read_text(encoding="utf-8"))
print(json.dumps(proof, indent=2))


{
  "status": "PASS",
  "counts": {
    "total_nodes": 57,
    "total_edges": 75,
    "distinct_node_ids": 57,
    "distinct_edge_ids": 75,
    "placeholder_nodes": 0
  },
  "node_breakdown": {
    "Load": 14,
    "Name": 11,
    "Constant": 4,
    "Expr": 3,
    "alias": 3,
    "arg": 3,
    "Call": 3,
    "arguments": 2,
    "ImportFrom": 2,
    "FunctionDef": 2,
    "Attribute": 2,
    "Return": 2,
    "Module": 1,
    "Import": 1,
    "JoinedStr": 1,
    "FormattedValue": 1,
    "Starred": 1,
    "keyword": 1
  },
  "edge_breakdown": {
    "AST": 56,
    "DFG": 9,
    "CFG": 7,
    "CALL": 3
  },
  "duplicate_node_ids": [],
  "duplicate_edge_ids": [],
  "connectors": {
    "neo4j-sink-cpg-nodes": {
      "connector": "RUNNING",
      "tasks": [
        "RUNNING"
      ]
    },
    "neo4j-sink-cpg-edges": {
      "connector": "RUNNING",
      "tasks": [
        "RUNNING"
      ]
    }
  }
}


## Bằng chứng Neo4j Browser

Ảnh dưới là truy vấn đúng `file_id` dùng trong Task 6. Tổng node/edge bằng số ID
distinct.

![Neo4j exact ID counts](images/task4/neo4j-exact-id-counts.png)

## Lỗi đã gặp và cách khắc phục

Worker ban đầu không ghi được plugin vào named volume; overlay chạy bước tải
connector với quyền phù hợp rồi Kafka Connect mới khởi động. Node và edge ở hai
topic nên có thể đến khác thời điểm; edge connector `MERGE` endpoint placeholder,
node connector điền đủ thuộc tính khi node event tới. Regression chờ exact ID
sets thay vì dựa vào sleep cố định.

## Reflection

`MERGE` và constraint giải quyết duplicate; DELETE event mới giải quyết stale
graph khi source thay đổi. Hai vấn đề này độc lập và đều phải được kiểm thử.

## Tái lập

```bash
bash task4/scripts/register_connectors.sh
python task4/verify_neo4j.py
curl http://localhost:8083/connectors/neo4j-sink-cpg-nodes/status
curl http://localhost:8083/connectors/neo4j-sink-cpg-edges/status
```
